# Capstone — Which visible pages should an editor review first?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROHITCRAFTSYT/flyrank-int/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

**Author:** Rohit Madhavan (GitHub [ROHITCRAFTSYT](https://github.com/ROHITCRAFTSYT)) · **Lane:** CTR / Engagement Opportunity Scoring · **Date:** 2026-08-02

This notebook mirrors my deployed research paper section-for-section, and **re-derives every number the paper quotes** from the in-repo starter slice (no token needed). The arc across the internship:

> A content team has more visible pages than review-hours. **Which page does an editor open first?**
> I score how far a page's clicks fall below *same-rank* peers, rank by it, and check — honestly — whether a
> learned model beats that rule. It doesn't (at one-week capacity), so the shipped tool is a transparent rule,
> turned into a human-reviewed action playbook with stated limits.

Run **Runtime → Run all**. Sections 1–6 print the paper's numbers; section 7 regenerates the figures the paper embeds. Claims stay *observed / measured / directional / decision-support*; nothing client-identifying is printed.

> Assistant note: read `skills/README.md`, then `writing-research-papers` + `deploying-static-pages` (the ML-11 card).

## 1. Question

**The decision.** A content editor or SEO lead has ~50 review-hours a week against thousands of already-visible pages. The decision is not *"is this page bad?"* but *"which page do I open on Monday?"* — a **ranking** problem, so the metric is **precision@K** at real review capacity (K≈50), never accuracy over the boring majority.

**Why there's signal to rank on.** Among pages with real exposure, CTR varies about **six-fold within a single position tier** (page-1: 0.12% at p25 vs 0.75% at p90). Position is a real but partial explanation (Spearman ≈ −0.18). That leftover — how far a page under-captures clicks *versus same-rank peers* — is exactly what an editor could act on.

**Careful words.** Everything here is *observed / directional / decision-support*. Never causal ("a rewrite will raise CTR"), never a claim about Google's algorithm, never a CTR level quoted outside this slice.

In [1]:
# --- Setup (runs in Colab or locally) + the numbers behind the question ------------
import os, sys, subprocess
import numpy as np, pandas as pd
pd.set_option("display.width", 180)
SEED = 0

IN_COLAB = "google.colab" in sys.modules
REPO_URL, REPO_DIR = "https://github.com/flyrank-bih/flyrank-ml-internship-starter", "flyrank-ml-internship-starter"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != os.path.dirname(os.getcwd()):
        os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv").drop_duplicates("content_id")

# My decision universe (ML-02): visible pages an editor could actually open this week.
# GOTCHA: avg_position == 0 means "no position data", not rank zero -> out of scope.
visible = df[(df.impressions_90d >= 500) & (df.avg_position > 0) & (df.avg_position <= 20) &
             (df.content_age_days >= 90)].copy()

# Number 1: CTR spread within a single position tier (rates are already in %, i.e. 0.24 = 0.24%).
p1 = visible.loc[visible.avg_position <= 10, "ctr"]          # page-1 pages
spread_x = p1.quantile(.90) / p1.quantile(.25)
# Number 2: position explains part, not all, of CTR.
spearman = visible[["avg_position", "ctr"]].corr(method="spearman").iloc[0, 1]

print(f"starter rows (dedup by content_id) : {len(df):,}")
print(f"MY DECISION UNIVERSE (visible)     : {len(visible):,} pages across {visible.client_id.nunique()} clients")
print(f"CTR spread within page 1 (p90/p25) : {spread_x:.1f}x  ({p1.quantile(.90):.2f}% vs {p1.quantile(.25):.2f}%)")
print(f"Spearman corr(avg_position, ctr)   : {spearman:.3f}  (real, negative, far from -1)")
print(f"\ncapacity squeeze: {len(visible):,} pages / 50 review-hours = a reviewer ever sees the top "
      f"{50/len(visible):.1%}  -> the ORDER is the product, so precision@K (not accuracy) is the metric")


starter rows (dedup by content_id) : 30,000
MY DECISION UNIVERSE (visible)     : 12,023 pages across 28 clients
CTR spread within page 1 (p90/p25) : 6.3x  (0.76% vs 0.12%)
Spearman corr(avg_position, ctr)   : -0.183  (real, negative, far from -1)

capacity squeeze: 12,023 pages / 50 review-hours = a reviewer ever sees the top 0.4%  -> the ORDER is the product, so precision@K (not accuracy) is the metric


## 2. Data

**Release.** The public, anonymized starter slice that ships with the FlyRank ML-internship repo: `data/raw/content_refresh_anonymized.csv` — **30,000 pseudonymized content items**, one row per `content_id`, with observed search/engagement metrics, content metadata, freshness fields, and 30-day comparison windows (`prev_30d` / `last_30d`). No client names, domains, URLs, page titles, or raw queries exist in the file (see `DATA_USE.md`).

**Date windows.** Metrics are a 90-day snapshot; the two nested 30-day windows (`prev_30d` = days 31–60, `last_30d` = days 1–30) are the only forward seam in the file and become the modeling label in §3.

**Exclusions (and why).** I keep only **visible** pages: ≥500 impressions/90d (enough exposure to rank on), average position 1–20 and >0 (a page with no position can't be compared to its tier; `avg_position==0` means *no data*, not rank zero), and ≥90 days old (excludes pages too new to judge). Pseudonymous IDs are used for **grouping/splitting only, never as features**.

In [2]:
# --- The exclusion waterfall + the columns I deliberately refuse as features -------
step = {}
step["0. starter rows (dedup)"]            = len(df)
step["1. impressions_90d >= 500"]          = len(df[df.impressions_90d >= 500])
step["2. + avg_position in (0, 20]"]       = len(df[(df.impressions_90d >= 500) & (df.avg_position > 0) & (df.avg_position <= 20)])
step["3. + content_age_days >= 90 (final)"] = len(visible)
print("=== exclusion waterfall (public-safe: counts only) ===")
for k, v in step.items():
    print(f"   {k:<34} {v:>7,}")
print(f"\n   -> visible universe = {len(visible)/len(df):.0%} of the starter slice, {visible.client_id.nunique()} clients")

# Columns that may NEVER be features (they encode the answer or a pseudonym).
LEAK_LIST = {
    "ctr / clicks_90d / impressions_90d": "the label is derived from CTR -> circular / outcome-window",
    "clicks_last_30d / ctr_last":         "the OUTCOME window itself",
    "trend_direction / trend_pct":        "ancestors of the starter's own label (data contract)",
    "content_id / client_id":             "pseudonyms: grouping & splitting only, never features",
}
print("\n=== leak list (excluded from every model on purpose) ===")
for col, why in LEAK_LIST.items():
    print(f"   - {col:<36} {why}")

# Prove the file carries nothing client-identifying.
UNSAFE = ["url", "domain", "title", "query", "keyword_text", "client_name"]
idcols = [c for c in df.columns if any(u in c.lower() for u in UNSAFE)]
print(f"\n   identifying columns in the file: {idcols or 'none'}  (public-safe by construction)")


=== exclusion waterfall (public-safe: counts only) ===
   0. starter rows (dedup)             30,000
   1. impressions_90d >= 500           16,726
   2. + avg_position in (0, 20]        12,023
   3. + content_age_days >= 90 (final)  12,023

   -> visible universe = 40% of the starter slice, 28 clients

=== leak list (excluded from every model on purpose) ===
   - ctr / clicks_90d / impressions_90d   the label is derived from CTR -> circular / outcome-window
   - clicks_last_30d / ctr_last           the OUTCOME window itself
   - trend_direction / trend_pct          ancestors of the starter's own label (data contract)
   - content_id / client_id               pseudonyms: grouping & splitting only, never features

   identifying columns in the file: none  (public-safe by construction)


## 3. Methodology

**The score (baseline / shipped rule).** For each position band, `expected_ctr` = the band's median CTR (the CTR-vs-rank curve). A page's shortfall is `missed_clicks_90d = max(0, expected_ctr − ctr) / 100 × impressions_90d`. Rank by it — that's the whole rule. Transparent, auditable, one line.

**The label (for the honest model race).** A single 90-day snapshot can't give a label the rule isn't already peeking at, so I use the one forward seam: decision moment = end of `prev_30d`, features strictly pre-decision, and an **observed** binary outcome measured strictly later — `y = ctr_last_30d > ctr_prev_30d` ("did CTR improve next month?"). Base rate ≈ **0.526**.

**Validation design (two guards at once).** (1) *Time-aware* — no feature reads the outcome window (the leak list). (2) *Grouped by client* — `GroupKFold(5)` keeps every client wholly on one side of each fold, so a model can't memorize a client. Both are asserted in code.

**Leakage checks.** The checklist from ML-09 passed: timeline order, no label-derived features, no product-decision flags, grouped split, base rate reported, top feature sane, out-of-fold scoring. Below I *demonstrate* the check working: feeding the model an outcome-window feature spikes AUC to ≈**0.999** — that's what leakage looks like, and it's why those columns are banned.

In [3]:
# --- (a) The score: CTR-vs-position curve + missed-clicks shortfall ----------------
visible["pos_band"]     = visible.avg_position.round().clip(1, 20).astype(int)
visible["expected_ctr"] = visible.groupby("pos_band")["ctr"].transform("median")   # the curve
visible["ctr_gap"]      = visible.ctr - visible.expected_ctr
visible["missed_clicks_90d"] = np.where(visible.ctr_gap < 0,
                                        (visible.expected_ctr - visible.ctr) / 100 * visible.impressions_90d, 0.0)
print("=== the shipped rule: expected CTR by position band (the curve every score rests on) ===")
curve = visible.groupby("pos_band")["ctr"].median().round(3)
print(curve.head(12).to_string())

# --- (b) The honest forward label + decision-point universe `d` (reused in section 4)
d = df[(df.impressions_prev_30d >= 200) & (df.clicks_prev_30d >= 1) &
       (df.avg_position > 0) & (df.avg_position <= 20)].copy()
d["ctr_prev"] = 100 * d.clicks_prev_30d / d.impressions_prev_30d
d["ctr_last"] = 100 * d.clicks_last_30d / d.impressions_last_30d.replace(0, np.nan)
d = d[d.ctr_last.notna()].copy()
d["y"] = (d.ctr_last > d.ctr_prev).astype(int)
base_rate = d.y.mean()
print(f"\ndecision-point universe : {len(d):,} pages / {d.client_id.nunique()} clients")
print(f"label y = ctr improved  : base rate {base_rate:.3f}  <- every scorer must beat this")

# --- (c) Feature matrix: pre-decision ONLY (the leak list is the whole game) --------
FEATURES = ["impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d", "ctr_prev",
            "word_count", "char_count", "content_age_days", "days_since_last_update",
            "search_volume", "competition", "cpc", "avg_position"]
def build_X(frame, cols):
    Xf = frame[cols].copy()
    for c in cols:
        Xf[c + "_na"] = Xf[c].isna().astype(int)
    return Xf.fillna(Xf.median(numeric_only=True))
X, y, groups = build_X(d, FEATURES), d.y.values, d.client_id.values

# --- (d) DEMONSTRATE the leakage check: an outcome-window feature spikes AUC --------
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score
gkf = GroupKFold(n_splits=5)
# overlap assert (grouped guard)
assert max(len(set(groups[tr]) & set(groups[te])) for tr, te in gkf.split(X, y, groups)) == 0
X_leak = X.copy(); X_leak["ctr_last_LEAK"] = d.ctr_last.values          # deliberately cheat
tr0, te0 = next(gkf.split(X, y, groups))
auc_clean = roc_auc_score(y[te0], GradientBoostingClassifier(random_state=SEED)
                          .fit(X.iloc[tr0], y[tr0]).predict_proba(X.iloc[te0])[:, 1])
auc_leak  = roc_auc_score(y[te0], GradientBoostingClassifier(random_state=SEED)
                          .fit(X_leak.iloc[tr0], y[tr0]).predict_proba(X_leak.iloc[te0])[:, 1])
print(f"\n=== leakage demonstration (one held-out client fold) ===")
print(f"   clean features        AUC = {auc_clean:.3f}   (honest)")
print(f"   + ctr_last (outcome)  AUC = {auc_leak:.3f}   <- ~0.999 is the smell of leakage; that column stays banned")


=== the shipped rule: expected CTR by position band (the curve every score rests on) ===
pos_band
1     0.060
2     0.215
3     0.300
4     0.330
5     0.290
6     0.240
7     0.220
8     0.190
9     0.200
10    0.170
11    0.190
12    0.180

decision-point universe : 8,348 pages / 28 clients
label y = ctr improved  : base rate 0.526  <- every scorer must beat this



=== leakage demonstration (one held-out client fold) ===
   clean features        AUC = 0.647   (honest)
   + ctr_last (outcome)  AUC = 0.998   <- ~0.999 is the smell of leakage; that column stays banned


## 4. Results (vs baseline)

Same universe, same split, same metric for every scorer. Headline **K = 50** (one reviewer's week); K = 100 shown because a scorer that wins at one K can lose at another — reporting both *is* the finding. Two floor baselines sit below the models on purpose: `naive_mean_reversion` (rank by lowest current CTR) measures how much of any score is just regression-to-the-mean; `baseline_ML07` is my shipped rule at the decision moment. Model scores are **out-of-fold**.

**The honest headline:** at P@50 the position-adjusted rule **ties** the best model (GradBoost) at **≈0.84 vs a 0.53 base rate** — so there's no case for shipping an opaque model at one-week capacity. The model only pulls ahead deeper in the queue (P@100), and most of the shared signal is mean reversion (`ctr_prev` dominates every scorer). *The simple thing winning is a result, not a failure.*

In [4]:
# --- Two baselines + four models, all scored out-of-fold on the same split ---------
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(labels)[order].mean())

# Baselines (no fitting): the ML-07 rule re-expressed at the decision moment + mean reversion.
d["pos_band"]     = d.avg_position.round().clip(1, 20).astype(int)
d["expected_ctr"] = d.groupby("pos_band")["ctr_prev"].transform("median")
d["missed_prev"]  = np.where(d.ctr_prev < d.expected_ctr,
                             (d.expected_ctr - d.ctr_prev) / 100.0 * d.impressions_prev_30d, 0.0)
base_scores = {"naive_mean_reversion": -d.ctr_prev.values, "baseline_ML07": d.missed_prev.values}

def make_models():
    return {
        "LogReg":          LogisticRegression(max_iter=2000, random_state=SEED),
        "DecisionTree_d3": DecisionTreeClassifier(max_depth=3, min_samples_leaf=50, random_state=SEED),
        "RandomForest":    RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=SEED, n_jobs=-1),
        "GradBoost":       GradientBoostingClassifier(random_state=SEED),
    }
model_names = list(make_models())
oof      = {m: np.full(len(d), np.nan) for m in model_names}
fold_auc = {m: [] for m in model_names}
for tr, te in gkf.split(X, y, groups):
    for name, mdl in make_models().items():
        if name == "LogReg":
            sc = StandardScaler().fit(X.iloc[tr])
            mdl.fit(sc.transform(X.iloc[tr]), y[tr]); p = mdl.predict_proba(sc.transform(X.iloc[te]))[:, 1]
        else:
            mdl.fit(X.iloc[tr], y[tr]); p = mdl.predict_proba(X.iloc[te])[:, 1]
        oof[name][te] = p
        if len(np.unique(y[te])) > 1:
            fold_auc[name].append(roc_auc_score(y[te], p))

rows = [("base_rate (random)", np.nan, base_rate, base_rate, base_rate)]
for name, s in base_scores.items():
    rows.append((name, np.nan, precision_at_k(s, y, 20), precision_at_k(s, y, 50), precision_at_k(s, y, 100)))
for m in model_names:
    rows.append((m, float(np.mean(fold_auc[m])),
                 precision_at_k(oof[m], y, 20), precision_at_k(oof[m], y, 50), precision_at_k(oof[m], y, 100)))
table = pd.DataFrame(rows, columns=["scorer", "AUC(grouped)", "P@20", "P@50", "P@100"])
print("=== MODEL vs BASELINE — same universe, same split, same metric ===")
print(f"    {len(d):,} pages / {d.client_id.nunique()} clients | label = CTR improved next 30d | base rate {base_rate:.3f}\n")
print(table.round(3).to_string(index=False))

best_model = max(model_names, key=lambda m: precision_at_k(oof[m], y, 50))
b07, mr = precision_at_k(base_scores["baseline_ML07"], y, 50), precision_at_k(base_scores["naive_mean_reversion"], y, 50)
print(f"\nP@50: baseline_ML07 {b07:.2f}  vs  best model {best_model} {precision_at_k(oof[best_model], y, 50):.2f}  "
      f"(mean-reversion floor {mr:.2f}); position-adjustment adds the real +{b07-mr:.2f} over the floor")


=== MODEL vs BASELINE — same universe, same split, same metric ===
    8,348 pages / 28 clients | label = CTR improved next 30d | base rate 0.526

              scorer  AUC(grouped)  P@20  P@50  P@100
  base_rate (random)           NaN 0.526 0.526  0.526
naive_mean_reversion           NaN 0.750 0.740  0.730
       baseline_ML07           NaN 0.800 0.840  0.730
              LogReg         0.643 0.650 0.660  0.700
     DecisionTree_d3         0.643 0.650 0.700  0.730
        RandomForest         0.654 0.950 0.780  0.780
           GradBoost         0.646 0.700 0.840  0.860

P@50: baseline_ML07 0.84  vs  best model GradBoost 0.84  (mean-reversion floor 0.74); position-adjustment adds the real +0.10 over the floor


## 5. Limitations

Each limit is *measured*, not asserted:

- **Not causal.** "Worth reviewing first" is decision-support; the data has no intervention, so I never claim a rewrite *causes* a CTR lift.
- **Position confound (the big one).** CTR can rise next month because a page *ranked* better, not because anyone improved it. The starter slice has no per-window rank to hold constant — the unconfounded verdict needs the warehouse's daily rank.
- **Small panel.** 28 clients — the grouped-split AUC gap (random vs client-grouped) is small but this is not a large or random sample of the web.
- **Sparse value.** CPC is known for only ~22% of the queue, and it's a benchmark, not booked revenue.
- **Thin position bands.** Bands 1–2 rest on few pages; `expected_ctr` there is shaky.
- **Not an SEO-algorithm claim.** This models one portfolio's outcomes; it says nothing about what Google rewards.

In [5]:
# --- Quantify the limits so the caveats carry numbers, not adjectives --------------
# Split sensitivity: how much does honesty (grouped) cost vs an over-optimistic random split?
from sklearn.model_selection import cross_val_predict, StratifiedKFold
gb_grouped = np.full(len(d), np.nan)
for tr, te in gkf.split(X, y, groups):
    gb_grouped[te] = GradientBoostingClassifier(random_state=SEED).fit(X.iloc[tr], y[tr]).predict_proba(X.iloc[te])[:, 1]
auc_grouped = roc_auc_score(y, gb_grouped)
gb_random = cross_val_predict(GradientBoostingClassifier(random_state=SEED), X, y,
                              cv=StratifiedKFold(5, shuffle=True, random_state=SEED), method="predict_proba")[:, 1]
auc_random = roc_auc_score(y, gb_random)

print("=== the limits, measured ===")
print(f"  universe coverage        : {len(visible):,} of {len(df):,} pages ({len(visible)/len(df):.0%}) meet the visible rule")
print(f"  clients (small panel)    : {d.client_id.nunique()}")
print(f"  CPC known (value usable) : {(d.cpc>0).mean():.0%} of pages; blank for the rest")
print(f"  split honesty cost (AUC) : random {auc_random:.3f} -> client-grouped {auc_grouped:.3f} "
      f"(gap {auc_random-auc_grouped:+.3f}; small = not heavily memorising clients)")
print(f"  position confound        : NOT fixable here (no per-window rank) -> unconfounded verdict needs the warehouse")
print(f"  age vs freshness         : median age {visible.content_age_days.median():.0f}d, "
      f"median days-since-update {visible.days_since_last_update.median():.0f}d -> 'old but fresh' (refresh is secondary here)")


=== the limits, measured ===
  universe coverage        : 12,023 of 30,000 pages (40%) meet the visible rule
  clients (small panel)    : 28
  CPC known (value usable) : 21% of pages; blank for the rest
  split honesty cost (AUC) : random 0.667 -> client-grouped 0.634 (gap +0.033; small = not heavily memorising clients)
  position confound        : NOT fixable here (no per-window rank) -> unconfounded verdict needs the warehouse
  age vs freshness         : median age 236d, median days-since-update 22d -> 'old but fresh' (refresh is secondary here)


## 6. Ranked recommendations

The validated part is the **queue order**; the shipped product is a human-reviewed **action playbook**. Every actionable page gets one archetype by a documented first-match cascade — the archetype's *action* is **directional guidance**, kept separate from the validated ranking:

| Archetype | Trigger | Action | Auto? |
|---|---|---|---|
| `verify_tracking` | CTR<0.03pp on ≥50k impressions (implausible) | Verify click tracking **before** any content work | **No — human only** |
| `stale_refresh` | under-capturing, age≥180d, ≥90d since update | Refresh facts/dates, expand thin sections | Draft only |
| `thin_expand` | under-capturing, word_count<1500 | Expand depth where demand exists | Draft only |
| `ctr_rewrite` | under-capturing, fresh + deep | Rewrite title & meta to lift CTR | Draft only |
| `striking_distance` | not under-capturing, position 11–20 | Improve relevance + internal links | Draft only |
| `monitor` | at/above expected, page-1, fresh | Watch; no action this cycle | n/a |

**Nothing auto-publishes.** Tracking-suspect pages are verified before any edit; the archetype action is never treated as a prediction; the queue is never ranked by dollar value alone (CPC is sparse). Recoverable clicks are **front-loaded** — the top 50 hold ~21% of all recoverable clicks — which is exactly why a capacity-limited team benefits from the ordering.

In [6]:
# --- Build the playbook queue (ML-10) on the visible + clicks>=1 universe -----------
vis = visible[visible.clicks_90d >= 1].copy()                         # playbook floor
vis["est_value_usd"] = (vis.missed_clicks_90d * vis.cpc.fillna(0)).round(2)   # clicks x CPC (never impressions)

ACTIONS = {"verify_tracking": "Verify click tracking BEFORE any content work",
           "stale_refresh": "Refresh: update facts/dates, expand thin sections",
           "thin_expand": "Expand depth where demand already exists",
           "ctr_rewrite": "Rewrite title & meta to lift CTR",
           "striking_distance": "Improve relevance + internal links toward page 1",
           "monitor": "No action this cycle; watch"}
def archetype(r):
    if r.ctr < 0.03 and r.impressions_90d >= 50_000:                 return "verify_tracking"
    if r.missed_clicks_90d <= 0:
        return "striking_distance" if r.avg_position >= 11 else "monitor"
    if r.content_age_days >= 180 and r.days_since_last_update >= 90:  return "stale_refresh"
    if pd.notna(r.word_count) and r.word_count < 1500:               return "thin_expand"
    return "ctr_rewrite"
vis["archetype"] = vis.apply(archetype, axis=1)

queue = (vis[vis.archetype != "monitor"]
         .sort_values(["missed_clicks_90d", "impressions_90d"], ascending=False).reset_index(drop=True))
cum_share = np.cumsum(queue.missed_clicks_90d.values) / queue.missed_clicks_90d.sum()

print(f"playbook universe : {len(vis):,} visible pages / {vis.client_id.nunique()} clients")
print(f"actionable queue  : {len(queue):,} pages ({(vis.archetype=='monitor').sum():,} held in 'monitor')\n")
print("=== queue composition by archetype (public-safe: counts + aggregates) ===")
comp = (vis.groupby("archetype").agg(pages=("content_id","size"),
        total_missed_clicks=("missed_clicks_90d","sum"),
        value_usd_where_cpc_known=("est_value_usd","sum"))
        .reindex(ACTIONS).fillna(0).round(0).astype({"pages":int}))
print(comp.to_string())
print(f"\ntotal recoverable clicks/90d : {vis.missed_clicks_90d.sum():,.0f}")
print(f"top-50 share of recoverable   : {cum_share[49]:.1%}  (front-loaded -> ordering pays off)")
print(f"value where CPC known (~{(vis.cpc>0).mean():.0%})   : ${vis.est_value_usd.sum():,.0f} (benchmark, not booked revenue)")


playbook universe : 10,808 visible pages / 28 clients
actionable queue  : 6,643 pages (4,165 held in 'monitor')

=== queue composition by archetype (public-safe: counts + aggregates) ===
                   pages  total_missed_clicks  value_usd_where_cpc_known
archetype                                                               
verify_tracking        8               1527.0                      506.0
stale_refresh       1271              15378.0                     5333.0
thin_expand           94                498.0                      346.0
ctr_rewrite         3310              37239.0                    18099.0
striking_distance   1960                  0.0                        0.0
monitor             4165                  0.0                        0.0

total recoverable clicks/90d : 54,642
top-50 share of recoverable   : 21.7%  (front-loaded -> ordering pays off)
value where CPC known (~22%)   : $24,284 (benchmark, not booked revenue)


## 7. Artifacts the paper embeds

The deployed page embeds four figures, all regenerated here into `work/figures/` so the paper and the notebook can never drift:

1. **CTR vs position** — the curve the whole rule rests on (trend + within-band spread).
2. **Model vs baseline (P@50 / P@100)** — the honest headline: the rule ties the model at K=50.
3. **Playbook composition by archetype** — what the queue is made of.
4. **Recoverable clicks vs queue depth** — why the ordering pays off (front-loaded).

A committed metrics receipt (`work/outputs/capstone_metrics.json`) records the numbers the paper quotes — safe aggregates only, no IDs.

In [7]:
# --- Regenerate the four figures the paper embeds + the committed metrics receipt ---
import json, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
os.makedirs("work/figures", exist_ok=True); os.makedirs("work/outputs", exist_ok=True)
GREEN, GREY = "#2f9e6f", "#9aa3ad"

# Fig 1 — CTR vs position (box per rounded band 1..10; median trend on top) ----------
bands = [b for b in range(1, 11) if (visible.pos_band == b).sum() >= 30]
data  = [visible.loc[visible.pos_band == b, "ctr"].values for b in bands]
fig, ax = plt.subplots(figsize=(7.2, 3.6))
bp = ax.boxplot(data, positions=bands, widths=0.6, showfliers=False, patch_artist=True)
for box in bp["boxes"]: box.set(facecolor="#dbeee4", edgecolor=GREEN)
for med in bp["medians"]: med.set(color="#1c6b4a", linewidth=1.5)
ax.plot(bands, [np.median(x) for x in data], color=GREEN, lw=2, marker="o", ms=3, label="median CTR")
ax.set_title("CTR varies widely within each position band — position isn't the whole story", fontsize=11)
ax.set_xlabel("average search position (rounded)"); ax.set_ylabel("CTR (%)"); ax.legend(frameon=False, fontsize=9)
for s in ["top", "right"]: ax.spines[s].set_visible(False)
fig.tight_layout(); fig.savefig("work/figures/paper_ctr_vs_position.png", dpi=120, bbox_inches="tight"); plt.close(fig)

# Fig 2 — Model vs baseline at P@50 and P@100 ---------------------------------------
order = ["base_rate (random)", "naive_mean_reversion", "baseline_ML07", best_model]
labels = ["random\n(base rate)", "naive mean\nreversion", "baseline\n(rule, shipped)", f"best model\n({best_model})"]
tt = table.set_index("scorer")
p50  = [tt.loc[s, "P@50"] for s in order]; p100 = [tt.loc[s, "P@100"] for s in order]
xpos = np.arange(len(order)); w = 0.38
fig, ax = plt.subplots(figsize=(7.2, 3.8))
b1 = ax.bar(xpos - w/2, p50, w, label="P@50 (one reviewer-week)", color=GREEN)
b2 = ax.bar(xpos + w/2, p100, w, label="P@100", color="#a7d3bf")
ax.axhline(base_rate, color=GREY, ls="--", lw=1); ax.text(len(order)-0.5, base_rate+0.01, "base rate", color="#666", fontsize=8, ha="right")
for bars in (b1, b2):
    for bar in bars: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f"{bar.get_height():.2f}", ha="center", fontsize=8)
ax.set_xticks(xpos); ax.set_xticklabels(labels, fontsize=9); ax.set_ylim(0, 1.0); ax.set_ylabel("precision@K")
ax.set_title("The rule ties the best model at K=50 — no case for shipping a black box", fontsize=11)
ax.legend(frameon=False, fontsize=9, loc="upper left")
for s in ["top", "right"]: ax.spines[s].set_visible(False)
fig.tight_layout(); fig.savefig("work/figures/paper_model_vs_baseline.png", dpi=120, bbox_inches="tight"); plt.close(fig)

# Fig 3 — Playbook composition by archetype -----------------------------------------
COLORS = {"verify_tracking": "#c0392b", "stale_refresh": "#8e6fb0", "thin_expand": "#5b8def",
          "ctr_rewrite": GREEN, "striking_distance": "#e0a63c", "monitor": GREY}
cp = vis.archetype.value_counts().reindex(list(ACTIONS)).fillna(0).astype(int)
fig, ax = plt.subplots(figsize=(7.2, 3.6))
ax.barh(cp.index[::-1], cp.values[::-1], color=[COLORS[a] for a in cp.index[::-1]])
for i, v in enumerate(cp.values[::-1]): ax.text(v + max(cp)*0.01, i, f"{v:,}", va="center", fontsize=9)
ax.set_title("Playbook queue composition by archetype (visible universe)", fontsize=11)
ax.set_xlabel("pages"); ax.margins(x=0.12)
for s in ["top", "right"]: ax.spines[s].set_visible(False)
fig.tight_layout(); fig.savefig("work/figures/w07_archetype_mix.png", dpi=120, bbox_inches="tight"); plt.close(fig)

# Fig 4 — Recoverable clicks vs queue depth (front-loaded) --------------------------
fig, ax = plt.subplots(figsize=(7.2, 3.6))
ax.plot(np.arange(1, len(cum_share)+1), cum_share, color=GREEN, lw=2)
for k in [50, 200, 500]:
    if k <= len(cum_share):
        ax.axvline(k, color=GREY, ls="--", lw=0.8)
        ax.text(k, 0.04, f" top {k}: {cum_share[k-1]:.0%}", rotation=90, va="bottom", fontsize=8, color="#555")
ax.set_title("Recoverable clicks are front-loaded: cumulative share vs queue depth", fontsize=11)
ax.set_xlabel("queue rank (pages reviewed, in order)"); ax.set_ylabel("share of total missed clicks"); ax.set_ylim(0, 1.02)
for s in ["top", "right"]: ax.spines[s].set_visible(False)
fig.tight_layout(); fig.savefig("work/figures/w07_review_depth_curve.png", dpi=120, bbox_inches="tight"); plt.close(fig)

# Committed metrics receipt (safe aggregates only, NO ids) --------------------------
capstone_metrics = {
    "task": "ML-11 capstone paper",
    "visible_universe_pages": int(len(visible)), "clients": int(visible.client_id.nunique()),
    "decision_universe_pages": int(len(d)), "base_rate": round(float(base_rate), 3),
    "precision_at_50": {r[0]: round(float(r[3]), 3) for r in rows},
    "precision_at_100": {r[0]: round(float(r[4]), 3) for r in rows},
    "grouped_auc_best": round(float(auc_grouped), 3), "random_auc_best": round(float(auc_random), 3),
    "leak_demo_auc_clean": round(float(auc_clean), 3), "leak_demo_auc_with_outcome": round(float(auc_leak), 3),
    "playbook_actionable_pages": int(len(queue)),
    "archetype_counts": {k: int((vis.archetype == k).sum()) for k in ACTIONS},
    "total_recoverable_clicks_90d": round(float(vis.missed_clicks_90d.sum()), 0),
    "top50_share_recoverable": round(float(cum_share[49]), 3),
    "value_usd_where_cpc_known": round(float(vis.est_value_usd.sum()), 2),
    "cpc_coverage": round(float((vis.cpc > 0).mean()), 3),
    "figures": ["paper_ctr_vs_position.png", "paper_model_vs_baseline.png",
                "w07_archetype_mix.png", "w07_review_depth_curve.png"],
    "claim_levels": {"queue_order": "validated decision-support (P@50 ~0.84 vs base ~0.53)",
                     "archetype_action": "directional guidance, not per-page validated"},
}
with open("work/outputs/capstone_metrics.json", "w") as fh:
    json.dump(capstone_metrics, fh, indent=2)
print("wrote 4 figures to work/figures/ and work/outputs/capstone_metrics.json")
print(json.dumps({k: capstone_metrics[k] for k in ["visible_universe_pages","base_rate","precision_at_50","top50_share_recoverable"]}, indent=2))


wrote 4 figures to work/figures/ and work/outputs/capstone_metrics.json
{
  "visible_universe_pages": 12023,
  "base_rate": 0.526,
  "precision_at_50": {
    "base_rate (random)": 0.526,
    "naive_mean_reversion": 0.74,
    "baseline_ML07": 0.84,
    "LogReg": 0.66,
    "DecisionTree_d3": 0.7,
    "RandomForest": 0.78,
    "GradBoost": 0.84
  },
  "top50_share_recoverable": 0.217
}


## Self-check

- [x] Every section filled — markdown reasoning AND the code that reproduces the number
- [x] Runs top to bottom with no errors (Runtime → Run all) — asserts guard the split; figures + metrics written
- [x] No client names, URLs, or private queries — only counts, aggregates, and pseudonymous grouping
- [x] Careful words throughout: *observed / measured / directional / decision-support*; queue-order vs archetype-action kept separate
- [x] Mirrors the deployed paper section-for-section; the four embedded figures regenerate here

**Deployed paper:** see `submission/paper_url.txt`. **Reproduce:** clone → `pip install -r requirements.txt` → Run all (seed 0).

### The finding in one honest paragraph
On this 28-client portfolio, in this period, I **observed** that visible pages under-capturing clicks relative to same-rank peers are the ones **worth reviewing first**, and a client-grouped, time-aware forward check **measured** that ranking at precision@50 ≈ 0.84 against a ≈0.53 base rate — a learned model did **not** beat it at one-week capacity, so the shipped tool is a transparent rule. It becomes a human-reviewed action playbook where nothing auto-publishes and the position confound is named, not hidden. **Decision-support for one portfolio — not a claim about what Google rewards.**